# Project: Fine-Tuning Llama 2 with QLoRA

**Course: LLMs with PyTorch — Capstone Project**

## Brief

Full fine-tuning of a 7-billion-parameter model needs many GPUs' worth of memory just
to hold the optimiser state. **QLoRA** (Quantized Low-Rank Adaptation) makes fine-tuning
a model like Llama 2 practical on a single consumer GPU by combining two ideas:

1. **4-bit quantization** — load the frozen base model's weights compressed to 4 bits
   instead of the usual 16, cutting memory roughly 4x with only a small quality cost
   (via the `bitsandbytes` library).
2. **LoRA (Low-Rank Adaptation)** — instead of updating all 7 billion weights, freeze
   the base model entirely and train a pair of small "adapter" matrices injected into
   each attention layer. Typically **under 1%** of the parameters actually get updated.

This notebook fine-tunes Llama 2 7B on a small instruction-following dataset using this
combination, via Hugging Face's `transformers`, `peft` (Parameter-Efficient Fine-Tuning)
and `trl` (Transformer Reinforcement Learning, which includes the supervised fine-tuning
trainer we use here) libraries.

> **Before you run this.** You need: (a) a GPU with at least ~16 GB VRAM, (b) the extra
> packages `peft`, `trl`, `bitsandbytes`, and `accelerate` installed
> (`pip install peft trl bitsandbytes accelerate`) on top of this repo's `transformers`
> and `torch`, and (c) a Hugging Face account with access approved for the gated
> `meta-llama/Llama-2-7b-hf` checkpoint. None of that is available in this sandbox, so
> the cells are written to be correct and runnable on suitable hardware, not executed
> here — read them for the QLoRA *workflow*, which is the transferable skill.

In [ ]:
# pip install -q -U transformers peft trl bitsandbytes accelerate datasets

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_NAME = "meta-llama/Llama-2-7b-hf"

## 1. The dataset

A tiny instruction-following dataset in the standard `instruction` / `response` shape
(the real project fine-tunes on a larger set of a few hundred to a few thousand
examples pulled from the Hugging Face Hub — this is a stand-in of the same shape you
can swap out). Each pair gets formatted into Llama 2's expected chat template before
training.

In [ ]:
raw_examples = [
    {"instruction": "Explain what LoRA does in one sentence.",
     "response": "LoRA freezes the pretrained model's weights and trains a small pair "
                 "of low-rank matrices injected into each layer, so fine-tuning updates "
                 "a tiny fraction of the parameters instead of all of them."},
    {"instruction": "What does QLoRA add on top of LoRA?",
     "response": "QLoRA loads the frozen base model in 4-bit precision before applying "
                 "LoRA adapters, cutting the memory needed to hold the base weights by "
                 "roughly 4x with a small, usually acceptable quality cost."},
    {"instruction": "Why freeze the base model at all?",
     "response": "Freezing the base model means you never need to store gradients or "
                 "optimizer state for its billions of parameters -- only for the small "
                 "adapter matrices -- which is what makes fine-tuning fit on one GPU."},
    {"instruction": "What is a rank in LoRA, in plain terms?",
     "response": "The rank controls the size of the adapter matrices: a small rank "
                 "(e.g. 8) means fewer trainable parameters and less capacity to adapt, "
                 "a larger rank means more of both -- it is a direct capacity/cost knob."},
]

def format_llama2_prompt(example):
    return {
        "text": f"<s>[INST] {example['instruction']} [/INST] {example['response']} </s>"
    }

dataset = Dataset.from_list(raw_examples).map(format_llama2_prompt)
print(dataset)
print()
print(dataset[0]["text"])

## 2. Loading the base model in 4-bit

`BitsAndBytesConfig` tells `transformers` to quantize the model as it loads, rather than
loading full-precision weights and quantizing afterwards. `nf4` ("normal float 4") is
the quantization scheme QLoRA's authors found to best preserve quality; the compute
dtype stays at `bfloat16` so the actual matrix multiplications during training are not
also degraded to 4-bit.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # a second quantization pass on the quant constants themselves
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",          # spread the model across whatever GPU(s) are visible
)
model = prepare_model_for_kbit_training(model)

## 3. Attaching LoRA adapters

`LoraConfig` specifies which layers get adapters (`target_modules` — the attention
projection matrices, where LoRA is usually most effective), the adapter rank `r`, and
`lora_alpha` (a scaling factor for the adapter's contribution, conventionally set to
`2 * r`). `get_peft_model` wraps the frozen base model with these trainable adapters.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

On a real 7B-parameter Llama 2, `print_trainable_parameters()` typically reports
something like *"trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%:
0.0622"* — well under 1% of the weights actually receive gradient updates. That number
is the entire point of the exercise: the same fine-tuning outcome as touching the whole
model, for a small fraction of the memory and compute.

## 4. Fine-tuning with `SFTTrainer`

`trl`'s `SFTTrainer` (Supervised Fine-Tuning Trainer) wraps the standard Hugging Face
`Trainer` with sensible defaults for instruction-tuning on a single `text` field —
tokenization, packing short examples together for efficiency, and the training loop
itself.

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama2-qlora-checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size 16, without needing the memory for it
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

trainer.train()

## 5. Saving and reloading just the adapter

Because the base model was never modified, you only need to save the small LoRA
adapter weights — typically a few megabytes, versus ~13 GB for the full model in
16-bit. Reloading means loading the base model once and attaching the adapter on top.

In [ ]:
model.save_pretrained("./llama2-car-assistant-adapter")
tokenizer.save_pretrained("./llama2-car-assistant-adapter")

import os
adapter_files = os.listdir("./llama2-car-assistant-adapter")
print("Saved adapter files:", adapter_files)

In [ ]:
from peft import PeftModel

# Reload: base model (still frozen, still 4-bit) + the fine-tuned adapter on top
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
fine_tuned_model = PeftModel.from_pretrained(base_model, "./llama2-car-assistant-adapter")
fine_tuned_model.eval()

## 6. Generating with the fine-tuned model

Same prompt format used during training (`[INST] ... [/INST]`), fed through
`generate()`. Compare the response quality here against the base model's response to
the same prompt to see what the fine-tuning actually changed.

In [ ]:
def ask(model, instruction, max_new_tokens=80):
    prompt = f"<s>[INST] {instruction} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(ask(fine_tuned_model, "What is the difference between LoRA and QLoRA?"))

## What to try next

* Swap in a larger, real instruction dataset (e.g. from the Hugging Face Hub) instead
  of the four toy examples used here — four examples are enough to prove the pipeline
  runs end to end, not enough to meaningfully change the model's behaviour.
* Add an evaluation step: hold out a few instructions, generate from both the base and
  fine-tuned models, and score the fine-tuned model's improvement with a rubric or a
  second LLM acting as judge.
* Experiment with the LoRA rank `r` — a higher rank gives the adapter more capacity at
  the cost of more trainable parameters and memory; this is the same capacity/overfitting
  trade-off covered for classical models in the Statistical Foundations module
  (Notebook 11, Overfitting), just applied to an adapter instead of a whole model.
* Try `bnb_4bit_quant_type="fp4"` instead of `"nf4"` and compare generation quality —
  the QLoRA paper found `nf4` better for weight distributions like a transformer's, but
  it is worth confirming empirically.